## HYQ-ALG-LIB 算法核心模块功能演示

欢迎来到 example_algorithm.ipynb。在前面的教程中，我们展示了如何使用 solvers 模块中的端到端算法（如 VQE、VarQITE 等）来求解分子的基态与激发态。本文件将带您深入 HYQ-ALG-LIB 的量子化学相关算法模块（algorithms 文件夹）。  
algorithms 模块是整个软件库的数学和理论骨架。它提供了构建高级量子算法所需的核心量子子程序、经典化学参考模型以及先进的测量与误差缓解协议。以下测试代码将逐一说明这个模块的具体功能，并验证这些底层模块的功能与特性。

### 1. 经典本征值求解器 classical_eigensolvers

#### 算法原理及特色
在开发量子算法时，获取绝对准确的经典对照组（即 Full Configuration Interaction, FCI 结果）是验证量子算法正确性的核心环节。该模块提供了两种求解哈密顿量矩阵的经典方法：

1. 精确对角化（Exact Diagonalization）：基于 NumPy 的密集矩阵特征值求解，适用于较小规模的希尔伯特空间。

2. Lanczos 迭代法：对于较大的量子比特系统，哈密顿量矩阵呈指数级增长。Lanczos 方法利用 SciPy 的稀疏矩阵运算，无需在内存中存储完整的哈密顿量矩阵，即可高效地提取出系统底部的几个极值本征态（如基态）。

#### 测试代码步骤与流程
1. 定义系统：代码首先手动定义了一个 3 量子比特的 Pauli 哈密顿量列表。

2. 执行精确对角化：调用 exact_diagonalization 函数，设置参数 k=3，提取前 3 个最低的本征值。

3. 执行 Lanczos 求解：调用 lanczos_lowest_eigenvalue 函数，独立计算该系统的最低本征值（基态能量）。

#### 具体验证了
1. 验证了不同形式的输入哈密顿量能够被正确解析并映射为 $2^n \times 2^n$ 的矩阵。
2. 验证了 dense-eigh 与 dense-eigh-lowest 两种底层执行路径的输出一致性，证明了经典求解器可以稳定地为后续量子化学计算提供严谨的基准能量。

In [1]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")  # 如果 notebook 在 examples/ 下
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
from hyq.algorithms.classical_eigensolvers import exact_diagonalization, lanczos_lowest_eigenvalue

hamiltonian = [
    (-1.0, "ZIX"),
    (-1.0, "IZY"),
    (0.5, "XXX"),
    (2.0, "YYX"),
    (1.5, "XYX"),
    (0.5, "XXZ"),
]

ed = exact_diagonalization(hamiltonian, n_qubits=3, k=3)
print(ed.method)
print(ed.eigenvalues)

lz = lanczos_lowest_eigenvalue(hamiltonian, n_qubits=3)
print(lz.method)
print(lz.eigenvalues)

Please first ``pip install -U qiskit`` to enable related functionality in translation module


dense-eigh
[-3.82866901 -2.97459248 -2.42526226]
dense-eigh-lowest
[-3.82866901]


### 2. 量子相位估计 phase_estimation.py
#### 算法原理及特色
量子相位估计（Quantum Phase Estimation, QPE）是提取幺正算符特征值的核心算法，广泛应用于 Shor 算法与早期量子化学算法中。目标是求解方程 $U|\psi\rangle = e^{2\pi i \phi}|\psi\rangle$ 中的相位 $\phi$。  

特色设计：模块不仅提供了生成 QPE 量子线路的构建器（包含 QFT 与逆 QFT 逻辑），还提供了一个纯经典的“精确有限寄存器概率分布模拟器”。该模拟器能够精确再现当目标相位无法被辅助比特二进制精确表示时，所产生的 Dirichlet 核泄漏（即概率幅分散）物理现象。

#### 测试代码步骤与流程
本节分为两个测试环节：
1. 经典概率分布模拟：设定目标相位 $\phi = 0.3$，使用 3 个辅助比特。由于 0.3 无法用 3 位二进制精确表示（最接近的是 0.25 和 0.375），代码调用 exact_phase_estimation_from_unitary 输出理论上的概率分布，并过滤打印出概率大于 0.001 的测量结果。

2. 量子线路真实模拟：设定目标相位 $\phi = 0.375$（恰好等于 $3/8$）。调用 build_qpe_circuit 真实构建包含 4 个量子比特的 QPE 线路。随后制备初始态，通过状态向量模拟器执行线路，最后调用 phases_from_counts 提取测量结果。

#### 具体验证了
第一段代码验证了模块对 QPE 固有误差（Dirichlet 泄漏）的经典模拟能力，输出显示最大概率集中在 0.25 和 0.375 附近，符合理论预期。第二、三段代码验证了 QPE 线路编译（包括控制幺正门和逆量子傅里叶变换）的完全正确性。因为 $0.375$ 可被 3 个辅助比特完美表示，模拟结果显示测得比特串 011 的概率达到了 100%。

In [ ]:
import numpy as np
from hyq.algorithms.phase_estimation import exact_phase_estimation_from_unitary

phi = 0.3
n_ancilla = 3

# |1> 是该 unitary 的 eigenstate，对应 eigenvalue exp(2πi phi)
U = np.diag([1.0, np.exp(2j * np.pi * phi)])
input_state = np.array([0.0, 1.0], dtype=np.complex128)

res = exact_phase_estimation_from_unitary(U, input_state, n_ancilla)

for bitstring, phase, prob in zip(res.bitstrings, res.phases, res.probabilities):
    if prob > 1e-3:
        print(bitstring, phase, prob)

000 0.0 0.02159321892578289
001 0.125 0.051768129535521595
010 0.25 0.5775210180698609
011 0.375 0.25933561918842774
100 0.5 0.0409067810742171
101 0.625 0.01944021679795832
110 0.75 0.014487479117612872
111 0.875 0.014947537290618543


In [ ]:
import numpy as np
from hyq.backends.core import QuantumCircuit
from hyq.algorithms.phase_estimation import build_qpe_circuit, phases_from_counts

phi = 0.375
n_ancilla = 3

# U |1> = exp(2πi phi) |1>
unitary = QuantumCircuit(1)
unitary.p(0, 2 * np.pi * phi)

qpe = build_qpe_circuit(unitary, n_ancilla=n_ancilla)

print(qpe)
print(qpe.stats())

<QuantumCircuit: qubits=4, instructions=16>
{'n_qubits': 4, 'n_instructions': 16, 'gate_counts': Counter({'Phase': 7, 'H': 6, 'CPhase': 3}), 'multi_qubit_gates': 10, 'depth_approx': 13, 'parameterized': True}


In [ ]:
import numpy as np
from hyq.backends._matrix_simulator import simulate_statevector, bitstring
from hyq.algorithms.phase_estimation import phases_from_counts

# QPE circuit 总 qubit 数 = 3 ancilla + 1 target = 4
# 初态 |000>|1>
initial_state = np.zeros(2**4, dtype=np.complex128)
initial_state[1] = 1.0

state = simulate_statevector(qpe, initial_state)

# 精确概率转成“伪 counts”
counts = {}
for idx, p in enumerate(np.abs(state) ** 2):
    if p > 1e-10:
        bits = bitstring(idx, qpe.n_qubits)
        ancilla_bits = bits[:n_ancilla]
        counts[ancilla_bits] = counts.get(ancilla_bits, 0.0) + float(p)

res = phases_from_counts(counts, n_ancilla)

print(counts)
print(res.bitstrings)
print(res.phases)
print(res.probabilities)

{'011': 1.0}
['011']
[0.375]
[1.]


### 3.量子子空间展开 qse

#### 算法原理及特色
量子子空间展开（Quantum Subspace Expansion, QSE）是一种先进的误差缓解与激发态求解技术。它在近似基态 $|\psi\rangle$ 的基础上，利用一组物理激发算符（如 $O_\mu$）构造扩展子空间，随后在该子空间内求解广义本征值问题 $H c = E S c$。  

特色设计：原生支持截断重叠矩阵（Overlap Cutoff）以剔除线性相关的冗余向量，避免广义本征值求解时出现数值奇异。内置了化学常用的单双激发池（Singles & Doubles Pool）生成工具。

#### 测试代码步骤与流程
代码执行了三个层次的测试：
1. 完备子空间测试：对于 1 量子比特系统，以 $|0\rangle$ 为参考态，使用 $X$ 算符进行激发。由于 $\{|0\rangle, X|0\rangle\}$ 覆盖了完整的希尔伯特空间，QSE 结果应与精确对角化完全一致。

2. 冗余算符与正则化测试：故意在激发池中放入 $I$, $2I$ 和 $X$。$I$ 和 $2I$ 产生的状态与参考态完全线性相关。

3. 化学算符池测试：针对 4 量子比特系统，构造了类 Hartree-Fock 参考态 $|1100\rangle$，并调用 singles_doubles_qse_pool 生成化学中常见的激发算符池进行展开。

#### 具体验证了
1. 验证了子空间哈密顿矩阵与重叠矩阵构建的数值正确性（测试 1 中的本征值与精确对角化完美吻合）。

2. 验证了算法的数值稳定性与冗余剔除机制。在测试 2 中，尽管输入了 3 个激发算符，算法通过阈值截断正确识别出有效子空间维度仅为 2。

3. 验证了化学激发池构建工具的格式兼容性，证明模块可以顺畅处理多量子比特的复杂 Pauli 字符串。

In [ ]:
import numpy as np
from hyq.algorithms.qse import quantum_subspace_expansion
from hyq.algorithms.classical_eigensolvers import exact_diagonalization

def basis_state(n_qubits, index):
    state = np.zeros(2 ** n_qubits, dtype=np.complex128)
    state[index] = 1.0
    return state

# H = Z + 0.2 X
hamiltonian = [
    (1.0, "Z"),
    (0.2, "X"),
]

reference_state = basis_state(1, 0)  # |0>

# 用 X|0> = |1> 扩展子空间，因此 QSE 子空间覆盖整个 1-qubit Hilbert space
excitation_terms = [
    [(1.0, "X")]
]

qse_result = quantum_subspace_expansion(
    reference_state=reference_state,
    hamiltonian=hamiltonian,
    excitation_terms=excitation_terms,
    n_qubits=1,
)

exact_result = exact_diagonalization(hamiltonian, n_qubits=1)

print("QSE energies:", qse_result.energies)
print("Exact energies:", exact_result.eigenvalues)
print("Overlap eigenvalues:", qse_result.overlap_eigenvalues)
print("Kept dimension:", qse_result.kept_subspace_dimension)

QSE energies: [-1.0198039  1.0198039]
Exact energies: [-1.0198039  1.0198039]
Overlap eigenvalues: [1. 1.]
Kept dimension: 2


In [ ]:
import numpy as np
from hyq.algorithms.qse import quantum_subspace_expansion

def basis_state(n_qubits, index):
    state = np.zeros(2 ** n_qubits, dtype=np.complex128)
    state[index] = 1.0
    return state

hamiltonian = [
    (1.0, "Z"),
    (0.2, "X"),
]

reference_state = basis_state(1, 0)

# I|0> 和 2I|0> 都和 reference state 线性相关
excitation_terms = [
    [(1.0, "I")],
    [(2.0, "I")],
    [(1.0, "X")],
]

result = quantum_subspace_expansion(
    reference_state=reference_state,
    hamiltonian=hamiltonian,
    excitation_terms=excitation_terms,
    n_qubits=1,
    regularization=1e-8,
)

print("QSE energies:", result.energies)
print("Overlap eigenvalues:", result.overlap_eigenvalues)
print("Kept dimension:", result.kept_subspace_dimension)

QSE energies: [-1.0198039  1.0198039]
Overlap eigenvalues: [-4.53155957e-16  9.06674729e-18  1.00000000e+00  3.00000000e+00]
Kept dimension: 2


In [ ]:
import numpy as np
from hyq.algorithms.qse import quantum_subspace_expansion, singles_doubles_qse_pool

def basis_state_from_bitstring(bitstring):
    n_qubits = len(bitstring)
    index = int(bitstring, 2)
    state = np.zeros(2 ** n_qubits, dtype=np.complex128)
    state[index] = 1.0
    return state

n_qubits = 4
n_electrons = 2

# Hartree-Fock-like |1100>
reference_state = basis_state_from_bitstring("1100")

hamiltonian = [
    (0.1, "ZIII"),
    (-0.2, "IZII"),
    (0.3, "IIZI"),
    (0.4, "IIIZ"),
    (0.05, "XXII"),
]

pool = singles_doubles_qse_pool(n_qubits, n_electrons)

result = quantum_subspace_expansion(
    reference_state=reference_state,
    hamiltonian=hamiltonian,
    excitation_terms=pool,
    n_qubits=n_qubits,
)

print("Lowest QSE energies:", result.energies[:5])
print("Subspace dimension:", result.kept_subspace_dimension)
print("First pool operator:", pool[0])

Lowest QSE energies: [-0.8        -0.40413813 -0.20413813  0.20413813  0.40413813]
Subspace dimension: 6
First pool operator: [(0.5, 'XIXI'), (0.5, 'YIYI')]


### 4.量子化学基础工具 quantum_chemistry

#### 算法原理及特色
此模块是一套诊断与后处理工具箱，无需运行复杂的变分量子算法即可提供丰富的物理化学信息。包括二阶 Møller-Plesset 微扰理论（MP2）校正、保真度计算、约化密度矩阵分析以及对称性检测。

特色设计：高度轻量化，输入接口兼容标准化学积分数组与底层 Pauli 字典，具备容错的归一化处理逻辑，保证即使在模拟器输出带有微小数值误差时也能返回可靠结果。

#### 测试代码步骤与流程
在一个集成测试块中连续调用了十余个核心函数：
1. 计算 FCI 参考能量。
2. 输入轨道能量和双电子积分，计算 MP2 相关能。
3. 计算两个态的 Fidelity 与相位。
4. 基于单体约化密度矩阵计算自然轨道占据数。
5. 评估未归一化波函数的总粒子数（$N$）和自旋投影（$S_z$）。
6. 对态执行对称性投影。
7. 提取活性空间索引与冻结核能量位移。
8. 计算算符对易子的矩阵范数。

#### 具体验证了
这部分代码验证了整个辅助工具链的可用性。它证明了模块能够正确处理：误差度量（例如将 Hartree 能量误差转换为 Kcal/mol 以判断是否达到化学精度）。量子态的物理守恒量分析（成功识别出状态的总粒子数为 2，总自旋为 0）。矩阵的代数性质提取（准确输出 $[X, Z]$ 对易子的范数数值）。

In [ ]:
import numpy as np

from hyq.algorithms.quantum_chemistry import (
    mp2_energy_from_integrals,
    fci_reference_energy,
    chemical_accuracy_error,
    state_fidelity,
    natural_orbital_occupations,
    particle_number_expectation,
    spin_z_expectation,
    active_space_indices,
    freeze_core_energy_shift,
    symmetry_project_state,
    commutator_norm,
)

# 1. FCI reference energy，使用库标准 Hamiltonian 格式
hamiltonian = [
    (1.0, "Z"),
    (0.2, "X"),
]

e_fci = fci_reference_energy(hamiltonian, n_qubits=1)
print("FCI energy:", e_fci)

err = chemical_accuracy_error(estimated_energy=-1.018, reference_energy=e_fci)
print(err)


# 2. MP2 reference energy
eps = np.array([0.0, 1.0])
eri = np.zeros((2, 2, 2, 2))

# chemist-order: eri[p,q,r,s] = (pq|rs)
eri[0, 1, 0, 1] = 0.2  # (0,1|0,1)

mp2 = mp2_energy_from_integrals(
    eps=eps,
    eri_mo=eri,
    n_electrons=1,
    hf_energy=-1.0,
)

print(mp2)
# MP2Result(correlation_energy=-0.02, total_energy=-1.02, ...)


# 3. Fidelity
state_a = np.array([1.0, 0.0], dtype=np.complex128)
state_b = np.array([1.0j, 0.0], dtype=np.complex128)

fid = state_fidelity(state_a, state_b)
print("Fidelity:", fid.fidelity)
print("Phase overlap:", fid.phase)


# 4. Natural orbital occupations
one_rdm = np.array([
    [1.8, 0.1],
    [0.1, 0.2],
], dtype=np.complex128)

occ = natural_orbital_occupations(one_rdm)
print("Natural occupations:", occ)


# 5. 粒子数和 Sz
state = np.zeros(4, dtype=np.complex128)
state[3] = 2.0  # 非归一化 |11>

print("N:", particle_number_expectation(state, n_qubits=2))
print("Sz:", spin_z_expectation(state, n_qubits=2))


# 6. 对称性投影
bell = np.array([1, 0, 0, 1], dtype=np.complex128) / np.sqrt(2)
projected = symmetry_project_state(
    bell,
    n_qubits=2,
    n_particles=2,
    sz=0.0,
)

print("Projected state:", projected)


# 7. Active space
active = active_space_indices(
    n_orbitals=10,
    n_core=2,
    n_active=4,
    n_virtual=1,
)
print("Active orbitals:", active)


# 8. Frozen-core scalar shift
shift = freeze_core_energy_shift(
    core_orbital_energies=[-20.0, -1.0],
    nuclear_repulsion=10.0,
    core_coulomb_exchange_shift=3.0,
)
print("Frozen-core shift:", shift)


# 9. Commutator norm
X = np.array([[0, 1], [1, 0]], dtype=np.complex128)
Z = np.array([[1, 0], [0, -1]], dtype=np.complex128)

print("||[X,Z]||:", commutator_norm(X, Z))

FCI energy: -1.019803902718557
{'hartree_error': 0.0018039027185570156, 'absolute_hartree_error': 0.0018039027185570156, 'kcal_per_mol_error': 1.1319660460688827, 'absolute_kcal_per_mol_error': 1.1319660460688827, 'within_chemical_accuracy': False}
MP2Result(correlation_energy=-0.020000000000000004, total_energy=-1.02, amplitudes_shape=(1, 1, 1, 1), skipped_denominators=0, convention='closed_shell_chemist')
Fidelity: 1.0
Phase overlap: 1j
Natural occupations: [1.80622577 0.19377423]
N: 2.0
Sz: 0.0
Projected state: [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
Active orbitals: [2, 3, 4, 5]
Frozen-core shift: -29.0
||[X,Z]||: 2.8284271247461903


### 5.哈密顿量模拟与 Trotter 分解 trotter

#### 算法原理及特色
模拟系统随时间的演化 $U(t) = e^{-i H t}$ 需要将其分解为量子计算机可执行的门序列。由于构成 $H$ 的各类 Pauli 算符互不对易，Trotter-Suzuki 公式提供了一种近似分解策略。  

特色设计：提供独立的一阶与二阶对称拆分逻辑。内置的 trotter_info 可以作为“编译器前端”，在无需消耗内存真实生成大型线路对象的前提下，快速估算出所需的量子门总数，方便评估 NISQ 硬件的可执行性。

#### 测试代码步骤与流程
1. 门数量估算：给定一个包含互不对易项的哈密顿量，调用 trotter_info 分别评估其一阶和二阶分解下的理论门数。

2. 线路生成与执行：真实调用 first_order_trotter_circuit 和 second_order_trotter_circuit 构造对应的 QuantumCircuit 对象，并打印线路的内部指令数以对照预估值。随后，将二阶线路置于状态向量模拟器中运行，观察态的演化结果。

#### 具体验证了
1. 验证了 Pauli 字符串演化模块的编译正确性，其内部正确插入了 CNOT 阶梯与中央 Rz 旋转。

2. 明确展示了不同阶数 Trotter 分解的资源开销对比（输出证明二阶对称拆分的门数严格为一阶的两倍，符合理论推导）。

In [ ]:
from hyq.algorithms.trotter import (
    first_order_trotter_circuit,
    second_order_trotter_circuit,
    trotter_info,
)

hamiltonian = [
    (-1.0, "ZI"),
    (-1.0, "IZ"),
    (0.5, "XX"),
]

time = 0.3
n_steps = 4
n_qubits = 2

qc1 = first_order_trotter_circuit(
    hamiltonian,
    time=time,
    n_steps=n_steps,
    n_qubits=n_qubits,
)

qc2 = second_order_trotter_circuit(
    hamiltonian,
    time=time,
    n_steps=n_steps,
    n_qubits=n_qubits,
)

print(qc1)
print(qc2)

print(trotter_info(hamiltonian, time=time, n_steps=n_steps, order=1))
print(trotter_info(hamiltonian, time=time, n_steps=n_steps, order=2))

Please first ``pip install -U qiskit`` to enable related functionality in translation module


<QuantumCircuit: qubits=2, instructions=36>
<QuantumCircuit: qubits=2, instructions=72>
TrotterStepInfo(order=1, time=0.3, n_steps=4, n_terms=3, gate_count_estimate=36)
TrotterStepInfo(order=2, time=0.3, n_steps=4, n_terms=3, gate_count_estimate=72)


In [ ]:
import numpy as np

from hyq.algorithms.trotter import second_order_trotter_circuit
from hyq.backends._matrix_simulator import simulate_statevector

hamiltonian = [
    (1.0, "X"),
    (0.7, "Z"),
]

qc = second_order_trotter_circuit(
    hamiltonian,
    time=0.8,
    n_steps=8,
    n_qubits=1,
)

initial_state = np.array([1.0, 0.0], dtype=np.complex128)
final_state = simulate_statevector(qc, initial_state)

print(final_state)

[ 5.60349564e-01-0.4760265j  -1.11022302e-16-0.67779579j]


In [ ]:
from hyq.algorithms.trotter import trotter_circuit

hamiltonian = [
    (-1.0, "ZI"),
    (-1.0, "IZ"),
    (0.5, "XX"),
]

qc = trotter_circuit(
    hamiltonian,
    time=1.0,
    n_steps=10,
    order=2,
    n_qubits=2,
)

print(qc)

<QuantumCircuit: qubits=2, instructions=180>


### 6.变分量子通缩 vqd

#### 算法原理及特色
在求得基态后，若需计算更高阶的激发态，变分量子通缩（VQD）提供了一种惩罚函数策略。通过向目标哈密顿量中注入正交约束项（重叠惩罚项），构造有效目标函数：$E_{\text{VQD}}(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle + \sum \beta_i |\langle\psi(\theta)|\phi_i\rangle|^2$。  
特色设计：模块同时支持直接从矩阵层面通缩（用于经典诊断）和基于状态向量重叠度的通缩计算。内置了 Gram-Schmidt 正交化工具，用于清理输入的简并或线性相关参考态集合。

#### 测试代码步骤与流程
1. 矩阵通缩验证：给定一个对角线为 $[0.0, 1.0, 2.0]$ 的三能级系统，已知基态为 $|0\rangle$。向其施加 $\beta=5.0$ 的惩罚项生成通缩后的矩阵，并对角化该新矩阵。

2. 目标函数评估：调用 vqd_objective_from_matrix，以第一激发态 $|1\rangle$ 作为候选态，评估通缩后的能量、惩罚项数值和重叠度。

3. 正交化处理：输入三个状态向量，其中前两个线性相关。调用 gram_schmidt_states 清理集合。

#### 具体验证了
第一段代码验证了通缩逻辑在矩阵代数层面的有效性。由于施加了足够大的惩罚系数 $\beta$，原始基态的本征值被抬升至 5，使得算法对角化后找到的“新基态”变为了真实的第一激发态（能量为 1.0），且与原始基态的重叠度严格为 0。第三段代码验证了 Gram-Schmidt 过程的鲁棒性，成功丢弃了线性相关的向量并输出了归一化的正交基。

In [ ]:
import numpy as np

from hyq.algorithms.vqd import (
    deflated_eigensystem,
    vqd_objective_from_matrix,
    state_overlap,
)

# 一个简单三能级 Hamiltonian
H = np.diag([0.0, 1.0, 2.0]).astype(np.complex128)

# 已经找到的基态 |0>
ground_state = np.array([1.0, 0.0, 0.0], dtype=np.complex128)

# beta 要大于目标能级间隔，才能把基态推高
betas = [5.0]

vals, vecs = deflated_eigensystem(
    H,
    previous_states=[ground_state],
    betas=betas,
)

print("Deflated eigenvalues:", vals)
print("Lowest deflated state:", vecs[:, 0])

# 检查最低 deflated state 是否正交于基态
print("Overlap with ground:", state_overlap(vecs[:, 0], ground_state))

Please first ``pip install -U qiskit`` to enable related functionality in translation module


Deflated eigenvalues: [1. 2. 5.]
Lowest deflated state: [0.+0.j 1.+0.j 0.+0.j]
Overlap with ground: 0.0


In [ ]:
import numpy as np
from hyq.algorithms.vqd import vqd_objective_from_matrix

H = np.diag([0.0, 1.0, 2.0]).astype(np.complex128)

ground_state = np.array([1.0, 0.0, 0.0], dtype=np.complex128)
candidate_state = np.array([0.0, 1.0, 0.0], dtype=np.complex128)

result = vqd_objective_from_matrix(
    H,
    candidate_state=candidate_state,
    previous_states=[ground_state],
    betas=[5.0],
)

print("Energy:", result.energy)
print("Overlaps:", result.overlaps)
print("Penalties:", result.penalties)
print("Total:", result.total)

Energy: 1.0
Overlaps: [0.0]
Penalties: [0.0]
Total: 1.0


In [ ]:
import numpy as np
from hyq.algorithms.vqd import gram_schmidt_states

states = [
    np.array([1.0, 0.0], dtype=np.complex128),
    np.array([2.0, 0.0], dtype=np.complex128),  # 和第一个线性相关，会被丢弃
    np.array([1.0, 1.0], dtype=np.complex128),
]

basis = gram_schmidt_states(states)

for v in basis:
    print(v)

[1.+0.j 0.+0.j]
[0.+0.j 1.+0.j]


### 7.经典阴影层析 shadow

#### 算法原理及特色
在 VQE 等变分算法中，传统方法需要对哈密顿量的每一项 Pauli 乘积进行独立的量子测量，开销极大。经典阴影（Classical Shadow）技术通过让线路随机在 X、Y、Z 基下进行局部测量收集“快照”，利用特殊的反演映射，能够在经典计算机上通过有限的测量次数重构量子态的全局信息。

特色设计：模块包含独立的数据收集器（ClassicalShadow 类）和估算器。用户可以通过它估计单个 Pauli 字符串、整个哈密顿量的期望，甚至能够反向推导重构整个体系的平均密度矩阵。

#### 测试代码步骤与流程
1. 单比特态验证：代码首先制备了单量子比特态 $|+i\rangle$（即 Y 基的 $+1$ 本征态）。然后定义了一个采样后端，调用 ClassicalShadow.collect() 收集 5000 个快照。接着，利用这些快照分别预测 X、Y、Z 和 I 的期望值，以及一个复合哈密顿量的总能量。

2. 多比特纠缠态验证：代码制备了两量子比特的 Bell 态 $\frac{|00\rangle + |11\rangle}{\sqrt{2}}$，收集 10000 个快照，用于预测两比特的 Pauli 期望值（如 XX、YY、ZZ）。

3. 密度矩阵重构：基于 Bell 态的快照，调用 estimate_density_matrix 重构出对应的 $4 \times 4$ 密度矩阵。

#### 具体验证了
对于单比特系统，输出清晰验证了算法的核心能力：在态 $|+i\rangle$ 下，测得的 $\langle Y \rangle$ 接近 1.0，$\langle X \rangle$ 和 $\langle Z \rangle$ 接近 0.0，符合量子力学理论。对于多比特 Bell 态，输出的估计值（$\langle XX \rangle \approx 1$, $\langle YY \rangle \approx -1$, $\langle ZZ \rangle \approx 1$）证明了经典阴影协议可以正确捕捉并重构量子比特之间的非局域纠缠关联性。密度矩阵的重构输出进一步验证了即使只使用随机局部测量，后处理模块也能通过逆信道映射恢复出系统的全局状态密度矩阵。

In [ ]:
import numpy as np

from hyq.backends.core import QuantumCircuit
from hyq.backends._matrix_simulator import simulate_statevector, sample_counts
from hyq.algorithms.shadow import ClassicalShadow


class NumpySamplingBackend:
    def __init__(self, seed=123):
        self.rng = np.random.default_rng(seed)

    def run_sampling(self, circuit, shots, measure_qubits):
        state = simulate_statevector(circuit)
        return sample_counts(
            state,
            shots=shots,
            measure_qubits=measure_qubits,
            seed=int(self.rng.integers(2**32)),
        )


# 制备 |+i> = S H |0>
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)

backend = NumpySamplingBackend(seed=1)

shadow = ClassicalShadow(
    backend=backend,
    circuit=qc,
    seed=2,
)

shadow.collect(5000)

print("<X> =", shadow.estimate_observable("X"))
print("<Y> =", shadow.estimate_observable("Y"))
print("<Z> =", shadow.estimate_observable("Z"))
print("<I> =", shadow.estimate_observable("I"))

Please first ``pip install -U qiskit`` to enable related functionality in translation module


<X> = 0.0054
<Y> = 0.996
<Z> = -0.009
<I> = 1.0


In [2]:
hamiltonian = [
    (1.0, "I"),
    (0.5, "Y"),
]

energy = shadow.estimate_hamiltonian(hamiltonian)
print("Estimated energy:", energy)

Estimated energy: (1.498+0j)


In [ ]:
from hyq.backends.core import QuantumCircuit
from hyq.algorithms.shadow import ClassicalShadow

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

shadow = ClassicalShadow(
    backend=NumpySamplingBackend(seed=3),
    circuit=qc,
    seed=4,
)

shadow.collect(10000)

print("<XX> =", shadow.estimate_observable("XX"))
print("<YY> =", shadow.estimate_observable("YY"))
print("<ZZ> =", shadow.estimate_observable("ZZ"))
print("<ZI> =", shadow.estimate_observable("ZI"))
print("<IZ> =", shadow.estimate_observable("IZ"))
print("<II> =", shadow.estimate_observable("II"))

<XX> = 0.9828
<YY> = -0.9855
<ZZ> = 0.9783
<ZI> = 0.0321
<IZ> = 0.024
<II> = 1.0


In [4]:
rho_hat = shadow.estimate_density_matrix()
print(rho_hat)

[[ 0.5086  +0.j       -0.002475+0.007425j -0.0069  -0.0024j
   0.492075+0.009225j]
 [-0.002475-0.007425j  0.00745 +0.j       -0.000675-0.008775j
   0.0012  +0.01155j ]
 [-0.0069  +0.0024j   -0.000675+0.008775j  0.0034  +0.j
   0.011025-0.004275j]
 [ 0.492075-0.009225j  0.0012  -0.01155j   0.011025+0.004275j
   0.48055 +0.j      ]]
